# BTC Autoresearch Experiment Analysis

Analysis of autonomous strategy optimization results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 7 columns)
df = pd.read_csv("results.tsv", sep="\t")
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
df["num_trades"] = pd.to_numeric(df["num_trades"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  score={row['score']:.6f}  sharpe={row['sharpe']:.2f}  acc={row['accuracy']:.3f}  trades={int(row['num_trades'])}  {row['description']}")

## Score Over Time

Track how the best (kept) score evolves as experiments progress. The running maximum shows the frontier -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_score = valid.loc[0, "score"]

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["score"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_scores = valid.loc[kept_mask, "score"]
running_max = kept_scores.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment
for idx, score in zip(kept_idx, kept_scores):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, score),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

best = kept_scores.max() if len(kept_scores) > 0 else baseline_score
n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Score (higher is better)", fontsize=12)
ax.set_title(f"BTC Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_score = df.iloc[0]["score"]
best_score = kept["score"].max()
best_row = kept.loc[kept["score"].idxmax()]

print(f"Baseline score:    {baseline_score:.6f}")
print(f"Best score:        {best_score:.6f}")
if baseline_score != 0:
    print(f"Total improvement: {best_score - baseline_score:.6f} ({(best_score - baseline_score) / abs(baseline_score) * 100:.1f}%)")
print(f"Best experiment:   {best_row['description']}")
print(f"Best sharpe:       {best_row['sharpe']:.4f}")
print(f"Best accuracy:     {best_row['accuracy']:.4f}")
print(f"Best num_trades:   {int(best_row['num_trades'])}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: score={row['score']:.6f}  sharpe={row['sharpe']:.2f}  acc={row['accuracy']:.3f}  {desc}")

## Top Hits (Kept Experiments by Score Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_score"] = kept["score"].shift(1)
kept["delta"] = kept["score"] - kept["prev_score"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>10}  {'Score':>10}  {'Sharpe':>8}  {'Acc':>6}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['score']:.6f}  {row['sharpe']:8.4f}  {row['accuracy']:.3f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>8}  {'':>6}  TOTAL improvement over baseline")